# Notebook 02: Treinamento — Mixed Precision Training e Gradient Checkpointing

## Aula 06: Aceleração com Hardware: GPU, TPU e Aceleradores

---

### Objetivos deste Notebook

1. Implementar **Mixed Precision Training** com AMP (Micikevicius et al., 2018)
2. Demonstrar **loss scaling dinâmico** com GradScaler para evitar underflow FP16
3. Aplicar **Gradient Checkpointing** para economia de memória (Chen et al., 2016)
4. Comparar performance entre FP32, FP16 e BF16 (quando disponível)
5. Visualizar curvas de treinamento e métricas de throughput

### Conexão com o Material Teórico

Este notebook implementa os **Snippets 1 e 3** do material da aula:

**Snippet 1 — Mixed Precision com AMP:**
```python
scaler = GradScaler()
with autocast(dtype=torch.float16):
    outputs = model(inputs)
    loss = criterion(outputs, targets)
```

**Snippet 3 — Gradient Checkpointing:**
```python
output = forward_with_checkpoint(transformer_block, hidden_states)
```

Conforme a seção **'Mixed Precision Training: Acelerando sem Comprometer'**:

> $$\text{Loss}_{\text{scaled}} = S \times \text{Loss}_{\text{original}}$$
> $$\nabla W_{\text{final}} = \frac{\nabla W_{\text{scaled}}}{S}$$

---

In [ ]:
# Imports
import sys
import os
import time
import logging

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

# Adicionar src/ ao path
sys.path.insert(0, os.path.abspath('..'))

from src.utils import get_device, get_hardware_info, set_seed, save_model, save_metrics
from src.data_preprocessing import DataPreprocessor
from src.model import SimpleViT, create_model
from src.training import Trainer, TrainingConfig, TrainingMetrics, benchmark_precision
from src.evaluation import plot_training_curves, plot_precision_comparison

# Configurações
logging.basicConfig(level=logging.INFO)
set_seed(42)
sns.set_theme(style='whitegrid')
%matplotlib inline

device = get_device()
print(f"Dispositivo: {device}")
print(f"CUDA disponível: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 1. Preparação dos Dados

Carregamos o dataset gerado no notebook anterior e criamos DataLoaders com
`batch_size=64` (múltiplo de 8 para alinhamento com Tensor Cores) e
`pin_memory=True` para transferência assíncrona CPU→GPU.

In [ ]:
# Carregar e preparar dados
preprocessor = DataPreprocessor(n_samples=10000, img_size=32, n_channels=3, n_classes=10, seed=42)
df = preprocessor.load_data()
df = preprocessor.clean_data(df)
features, labels = preprocessor.prepare_features(df)
splits = preprocessor.split_data(features, labels)

# DataLoaders com configuração otimizada para GPU
BATCH_SIZE = 64  # Múltiplo de 8 para Tensor Cores
loaders = preprocessor.create_dataloaders(
    splits, batch_size=BATCH_SIZE,
    pin_memory=torch.cuda.is_available(),
)

print(f"Splits: {', '.join(f'{k}: {v[0].shape[0]}' for k, v in splits.items())}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Batches por época (train): {len(loaders['train'])}")

## 2. Definição do Modelo: Vision Transformer

Conforme mencionado no material da aula:

> "Trabalharemos com um **Vision Transformer** como caso de estudo, analisando 
> operações compute-bound e memory-bound."

O modelo usa `embed_dim=128` (múltiplo de 64) para alinhamento com Tensor Cores:

> "Escolher dimensões de embedding múltiplas de 64 garante utilização ótima em 
> todas as gerações de Tensor Cores" — seção 'Tensor Cores' do material da aula.

In [ ]:
# Criar Vision Transformer — caso de estudo da aula
model = create_model(
    img_size=32,
    patch_size=4,       # 32/4 = 8x8 = 64 patches
    in_channels=3,
    n_classes=10,
    embed_dim=128,      # Múltiplo de 64 para Tensor Cores
    depth=4,            # 4 blocos Transformer
    n_heads=4,          # head_dim = 128/4 = 32 (múltiplo de 8)
    use_checkpointing=False,
)

# Inspecionar modelo
memory = model.get_memory_footprint()
print(f"\n=== MODELO ===")
print(f"  Parâmetros:  {memory['params_count']:,}")
print(f"  Memória FP32: {memory['fp32_mb']:.2f} MB")
print(f"  Memória FP16: {memory['fp16_mb']:.2f} MB")
print(f"  Memória INT8: {memory['int8_mb']:.2f} MB")
print(f"\n  Estrutura:")
print(f"  {model}")

## 3. Treinamento Baseline (FP32)

Primeiro, treinamos em **FP32 puro** como baseline para comparação com mixed precision.
Conforme Micikevicius et al. (2018), o treinamento FP32 é o padrão convencional
que serve de referência para medir os ganhos de mixed precision.

In [ ]:
# Treinamento FP32 (baseline)
# Sem AMP — usa precisão completa para todas operações

model_fp32 = create_model(embed_dim=128, depth=4, n_heads=4, use_checkpointing=False)

config_fp32 = TrainingConfig(
    n_epochs=5,
    learning_rate=1e-3,
    batch_size=BATCH_SIZE,
    use_amp=False,          # Sem Mixed Precision
    use_checkpointing=False,
    seed=42,
)

trainer_fp32 = Trainer(model_fp32, config_fp32, device)

print("=== TREINAMENTO FP32 (BASELINE) ===")
start_time = time.time()
metrics_fp32 = trainer_fp32.train(loaders['train'], loaders['val'])
total_time_fp32 = time.time() - start_time

print(f"\nTempo total FP32: {total_time_fp32:.2f}s")
print(f"Throughput médio: {np.mean(metrics_fp32.throughputs):.0f} amostras/s")

## 4. Treinamento com Mixed Precision (FP16 + AMP)

Agora implementamos **Mixed Precision Training** conforme **Snippet 1** do material:

```python
# Configuração do GradScaler para loss scaling dinâmico
scaler = GradScaler()

# Loop de treinamento com mixed precision
with autocast(dtype=torch.float16):
    outputs = model(inputs)
    loss = criterion(outputs, targets)
```

O **GradScaler** implementa loss scaling dinâmico:

$$\text{Loss}_{\text{scaled}} = S \times \text{Loss}_{\text{original}}$$

> "O GradScaler ajusta automaticamente o fator de escala do loss para evitar 
> underflow em gradientes FP16" — material da aula, Snippet 1.

In [ ]:
# Treinamento com AMP (FP16) — Snippet 1 do material da aula
# autocast converte forward/backward para FP16
# GradScaler aplica loss scaling dinâmico (Micikevicius et al., 2018)

model_amp = create_model(embed_dim=128, depth=4, n_heads=4, use_checkpointing=False)

config_amp = TrainingConfig(
    n_epochs=5,
    learning_rate=1e-3,
    batch_size=BATCH_SIZE,
    use_amp=True,           # Ativa Mixed Precision
    amp_dtype=torch.float16,  # FP16 com loss scaling
    use_checkpointing=False,
    seed=42,
)

trainer_amp = Trainer(model_amp, config_amp, device)

print("=== TREINAMENTO MIXED PRECISION (FP16 + AMP) ===")
start_time = time.time()
metrics_amp = trainer_amp.train(loaders['train'], loaders['val'])
total_time_amp = time.time() - start_time

print(f"\nTempo total AMP: {total_time_amp:.2f}s")
print(f"Throughput médio: {np.mean(metrics_amp.throughputs):.0f} amostras/s")
if total_time_fp32 > 0:
    speedup = total_time_fp32 / total_time_amp
    print(f"Speedup vs FP32: {speedup:.2f}x")

## 5. Gradient Checkpointing — Economia de Memória

Implementa o **Snippet 3** do material da aula:

```python
from torch.utils.checkpoint import checkpoint
output = forward_with_checkpoint(transformer_block, hidden_states)
```

Conforme a seção **'Otimização de Memória GPU: Gradient Checkpointing'** (Chen et al., 2016):

> "Ao invés de armazenar todas as ativações intermediárias para o backward pass,
> o método armazena apenas checkpoints periódicos, recomputando ativações intermediárias 
> conforme necessário."

- Memória convencional: **O(n)** com profundidade n
- Com checkpointing: **O(√n)** ao custo de ~33% computação adicional

In [ ]:
# Treinamento com Gradient Checkpointing — Snippet 3 (Chen et al., 2016)
# Reduz memória de O(n) para O(√n) ao custo de ~33% computação adicional

model_ckpt = create_model(
    embed_dim=128, depth=4, n_heads=4,
    use_checkpointing=True,  # Ativa gradient checkpointing
)

config_ckpt = TrainingConfig(
    n_epochs=5,
    learning_rate=1e-3,
    batch_size=BATCH_SIZE,
    use_amp=True,
    amp_dtype=torch.float16,
    use_checkpointing=True,  # Flag para referência
    seed=42,
)

trainer_ckpt = Trainer(model_ckpt, config_ckpt, device)

print("=== TREINAMENTO COM AMP + GRADIENT CHECKPOINTING ===")
start_time = time.time()
metrics_ckpt = trainer_ckpt.train(loaders['train'], loaders['val'])
total_time_ckpt = time.time() - start_time

print(f"\nTempo total AMP+Ckpt: {total_time_ckpt:.2f}s")
print(f"Throughput médio: {np.mean(metrics_ckpt.throughputs):.0f} amostras/s")

## 6. Comparação de Resultados

Conforme documentado por Micikevicius et al. (2018):

> "Treinamento de BERT-large com mixed precision FP16 reduz tempo de treinamento 
> em 1.8x com degradação de acurácia inferior a 0.1%."

> "ResNet-50 em ImageNet alcança convergência equivalente com speedup de 2.8x 
> em GPUs V100 equipadas com Tensor Cores."

In [ ]:
# Comparação detalhada das três configurações
print("=" * 80)
print("COMPARAÇÃO DE CONFIGURAÇÕES DE TREINAMENTO")
print("=" * 80)

configs = {
    'FP32 (baseline)': {'metrics': metrics_fp32, 'time': total_time_fp32},
    'AMP FP16': {'metrics': metrics_amp, 'time': total_time_amp},
    'AMP FP16 + Ckpt': {'metrics': metrics_ckpt, 'time': total_time_ckpt},
}

print(f"{'Configuração':25s} {'Loss Final':>10s} {'Acc Train':>10s} {'Acc Val':>10s} "
      f"{'Tempo (s)':>10s} {'Speedup':>8s} {'Mem (MB)':>10s}")
print("-" * 90)

for name, data in configs.items():
    m = data['metrics']
    t = data['time']
    speedup = total_time_fp32 / t if t > 0 else 0
    last_train_acc = m.train_accuracies[-1] if m.train_accuracies else 0
    last_val_acc = m.val_accuracies[-1] if m.val_accuracies else 0
    peak_mem = max(m.peak_memory_mb) if m.peak_memory_mb else 0
    
    print(f"  {name:23s} {m.train_losses[-1]:10.4f} {last_train_acc:10.4f} "
          f"{last_val_acc:10.4f} {t:10.2f} {speedup:7.2f}x {peak_mem:10.1f}")

In [ ]:
# Visualização: curvas de treinamento comparadas
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

colors = {'FP32 (baseline)': '#2196F3', 'AMP FP16': '#4CAF50', 'AMP FP16 + Ckpt': '#FF9800'}

for name, data in configs.items():
    m = data['metrics']
    epochs = range(1, len(m.train_losses) + 1)
    color = colors[name]
    
    axes[0, 0].plot(epochs, m.train_losses, '-o', color=color, label=name, markersize=4)
    if m.val_losses:
        axes[0, 1].plot(epochs, m.val_losses, '-o', color=color, label=name, markersize=4)
    axes[1, 0].plot(epochs, m.throughputs, '-o', color=color, label=name, markersize=4)
    if m.peak_memory_mb and any(v > 0 for v in m.peak_memory_mb):
        axes[1, 1].plot(epochs, m.peak_memory_mb, '-o', color=color, label=name, markersize=4)

axes[0, 0].set_title('Training Loss'); axes[0, 0].set_xlabel('Época'); axes[0, 0].set_ylabel('Loss')
axes[0, 1].set_title('Validation Loss'); axes[0, 1].set_xlabel('Época'); axes[0, 1].set_ylabel('Loss')
axes[1, 0].set_title('Throughput (amostras/s)'); axes[1, 0].set_xlabel('Época'); axes[1, 0].set_ylabel('Amostras/s')
axes[1, 1].set_title('Pico de Memória GPU (MB)'); axes[1, 1].set_xlabel('Época'); axes[1, 1].set_ylabel('MB')

for ax in axes.flat:
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

plt.suptitle('Comparação: FP32 vs AMP FP16 vs AMP+Checkpointing\n(Micikevicius et al., 2018; Chen et al., 2016)',
             fontsize=13)
plt.tight_layout()
plt.savefig('../outputs/figures/comparacao_treinamento.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Benchmark de Inferência por Precisão

Benchmarkamos o modelo em diferentes precisões conforme discutido na seção
**'Mixed Precision Training: Acelerando sem Comprometer'**:

- **FP32:** Baseline de precisão completa
- **FP16:** Half precision com Tensor Cores (speedup ~2x esperado)
- **BF16:** Brain Float 16, mesmo range do FP32 (Kalamkar et al., 2019)

In [ ]:
# Benchmark de inferência em diferentes precisões
# Compara FP32, FP16 e BF16 conforme seção sobre Mixed Precision

model_bench = create_model(embed_dim=128, depth=4, n_heads=4)
input_shape = (BATCH_SIZE, 3, 32, 32)

print("\n=== BENCHMARK DE PRECISÕES NUMÉRICAS ===")
bench_results = benchmark_precision(
    model_bench, input_shape, device,
    n_iterations=50, warmup=10,
)

print(f"\n{'Precisão':>10s} {'Tempo (ms)':>12s} {'Std (ms)':>10s} {'Throughput':>15s} {'Speedup':>10s}")
print("-" * 60)
base_time = bench_results.get('fp32', {}).get('avg_time_ms', 1)
for name, res in bench_results.items():
    speedup = base_time / res['avg_time_ms'] if res['avg_time_ms'] > 0 else 0
    print(f"  {name.upper():>8s} {res['avg_time_ms']:10.2f} ms {res['std_time_ms']:8.2f} ms "
          f"{res['throughput_samples_per_sec']:>13.0f}/s {speedup:8.2f}x")

In [ ]:
# Visualização do benchmark de precisões
fig = plot_precision_comparison(
    bench_results,
    save_path='../outputs/figures/benchmark_precisoes.png',
    title='Comparação de Precisões Numéricas (Micikevicius et al., 2018)',
)

# Recriar para display
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
names = list(bench_results.keys())
colors_map = {'fp32': '#2196F3', 'fp16': '#4CAF50', 'bf16': '#FF9800'}
bar_colors = [colors_map.get(n, '#9E9E9E') for n in names]

times = [bench_results[n]['avg_time_ms'] for n in names]
axes[0].bar([n.upper() for n in names], times, color=bar_colors)
axes[0].set_ylabel('Tempo por Batch (ms)')
axes[0].set_title('Latência por Precisão')
for i, v in enumerate(times):
    axes[0].text(i, v * 1.02, f'{v:.2f}', ha='center', fontsize=10)

tputs = [bench_results[n]['throughput_samples_per_sec'] for n in names]
axes[1].bar([n.upper() for n in names], tputs, color=bar_colors)
axes[1].set_ylabel('Throughput (amostras/s)')
axes[1].set_title('Throughput por Precisão')
for i, v in enumerate(tputs):
    axes[1].text(i, v * 1.02, f'{v:.0f}', ha='center', fontsize=10)

plt.suptitle('Impacto de Mixed Precision na Performance\n(Micikevicius et al., 2018)', fontsize=13)
plt.tight_layout()
plt.show()

## 8. Salvar Modelo e Métricas

Salvamos o melhor modelo (AMP FP16) para uso no notebook de avaliação.

In [ ]:
# Salvar o melhor modelo (AMP) e métricas
from src.utils import save_model, save_metrics

# Modelo
save_model(
    model_amp,
    metadata={
        'config': 'AMP FP16',
        'epochs': 5,
        'final_train_loss': metrics_amp.train_losses[-1],
        'final_val_acc': metrics_amp.val_accuracies[-1] if metrics_amp.val_accuracies else None,
    }
)

# Métricas
all_metrics = {
    'fp32': {
        'train_losses': metrics_fp32.train_losses,
        'val_losses': metrics_fp32.val_losses,
        'train_accuracies': metrics_fp32.train_accuracies,
        'val_accuracies': metrics_fp32.val_accuracies,
        'throughputs': metrics_fp32.throughputs,
        'total_time': total_time_fp32,
    },
    'amp_fp16': {
        'train_losses': metrics_amp.train_losses,
        'val_losses': metrics_amp.val_losses,
        'train_accuracies': metrics_amp.train_accuracies,
        'val_accuracies': metrics_amp.val_accuracies,
        'throughputs': metrics_amp.throughputs,
        'total_time': total_time_amp,
    },
    'amp_ckpt': {
        'train_losses': metrics_ckpt.train_losses,
        'val_losses': metrics_ckpt.val_losses,
        'train_accuracies': metrics_ckpt.train_accuracies,
        'val_accuracies': metrics_ckpt.val_accuracies,
        'throughputs': metrics_ckpt.throughputs,
        'total_time': total_time_ckpt,
    },
    'benchmark_precision': bench_results,
}

save_metrics(all_metrics, filename='training_metrics.json')
print("✓ Modelo e métricas salvos em outputs/")

## Resumo

Neste notebook, implementamos e comparamos:

1. **FP32 (baseline):** Treinamento convencional em precisão completa
2. **AMP FP16:** Mixed Precision Training com loss scaling dinâmico (Micikevicius et al., 2018)
3. **AMP + Checkpointing:** Combinação de mixed precision com gradient checkpointing (Chen et al., 2016)

### Observações Esperadas (em GPU com Tensor Cores)

- **Speedup AMP vs FP32:** ~1.5-2.8x (conforme Micikevicius et al., 2018)
- **Redução de memória com Checkpointing:** até 60% em modelos grandes
- **Degradação de acurácia:** < 0.1% (AMP mantém master weights em FP32)

> **Nota:** Em CPU, o AMP não traz speedup significativo pois não há Tensor Cores.
> Os ganhos são observados em GPUs NVIDIA com compute capability ≥ 7.0.

---

**Próximo notebook:** [03_avaliacao.ipynb](03_avaliacao.ipynb) — Profiling, avaliação e análise de gargalos